# **MLLM-SHAP: Практика**

Привет, друзья! В уроке «Как убирать признаки на разных модальностях? Эффекты масок» мы разобрали SGPA: аудио режут на слова, выключенное слово заменяют тишиной, а значения Шепли оценивают комплементарными вкладами $CC(S)=v(S)-v(N\setminus S)$ с неймановским распределением выборки по размерам коалиций. В уроке «SHAP на генеративных моделях» добавилась вторая мысль: когда выход модели не число, $v(S)$ приходится задавать сравнением ответов, и выбор функции ценности есть выбор самой игры.

Сегодня проверим обе мысли на библиотеке [`mllm-shap`](https://github.com/Pawlo77/MLLM-SHAP) (Paweł Pozorski, [документация](https://pawlo77.github.io/MLLM-SHAP/)), в которой реализованы и SGPA, и оценщики из урока. Объяснять будем маленькую открытую чат-модель `Qwen2.5-0.5B-Instruct`: она идет на CPU Colab, а ключ API и токен Hugging Face ей не нужны.

В этой практике вы:
- посчитаете точные значения Шепли для слов промпта оценщиком `exact` и проверите эффективность: $\sum_i\phi_i = v(N)-v(\varnothing)$;
- соберете таблицу $v(S)$ всех 128 коалиций и сами пересчитаете по ней значения Шепли;
- сравните с точным ответом три приближенных оценщика — Монте-Карло, комплементарный и комплементарный с неймановским распределением — при разных бюджетах прогонов;
- посмотрите, как меняется атрибуция при смене функции ценности, и зададите свою: лог-вероятность ответа «Paris»;
- разрежете запись речи на слова через SGPA и выключите часть слов тишиной;
- соберете PixelSHAP своими руками, с объектами картинки как игроками маленькой VLM, и проверите, насколько точно приближение первого порядка.

Приступим!

Сначала зависимости. На Colab нет только `mllm-shap`; версию закрепим, чтобы числа совпадали с нашими. `torch`, `transformers`, `librosa` и `soundfile` на Colab уже стоят. Дополнение `mllm-shap[viz]` ставить не нужно: оно требует `ipywidgets>=8.1` и обновило бы версию Colab, а графики библиотеки рисуются обычным `matplotlib`.

In [ ]:
!pip install mllm-shap==0.3.0 -q

In [ ]:
import random
import urllib.request
from itertools import combinations
from math import comb, factorial

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from transformers import AutoModelForCausalLM, AutoTokenizer

import mllm_shap
from mllm_shap import (AudioPart, CoalitionMask, ConversationSnapshot, DeletePolicy, ExplainConfig,
                       ExplainRunner, Message, NeutralPolicy, PluginContext, Role, SgpaSegmentPartitioner,
                       TextPart, TransformersTextBackend, Turn)
from mllm_shap.alignment import SpectrogramGuidedAligner, Wav2Vec2Ctc, decode_audio
from mllm_shap.viz import plot_audio_segments


def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('mllm-shap', mllm_shap.__version__, '| device:', device)

## Игра: слова промпта и ответ модели

В `mllm-shap` объяснение собирается из взаимозаменяемых частей, и каждая часть — одно решение из уроков:

| часть конфигурации | что задает | что берем сегодня |
|---|---|---|
| `players` | кто игроки | `tokens` — слова сообщения пользовательницы (разбиение по пробелам) |
| `absence_policy` | как «выключить» игрока | `delete` — слово удаляется из строки |
| `value_fn` | функция ценности $v(S)$ | `logprob`, потом `tfidf_cosine` и `embedding_cosine` |
| `estimator` | как перебирать коалиции | `exact`, `mc`, `complementary`, `neyman_cc` |
| `backend` | чья модель отвечает | `transformers` — локальная модель Hugging Face |

Разговор такой: системное сообщение «Answer with one word.» и вопрос пользовательницы «What is the capital city of France?». Игроки — 7 слов вопроса, системное сообщение присутствует во всех коалициях. Генерация жадная (`temperature=0`, `top_k=1`), не длиннее 6 токенов, поэтому один и тот же урезанный промпт всегда дает один и тот же ответ. Ревизию модели на Hugging Face закрепим хешем коммита.

Две технические детали. Бэкенд `transformers` сам выбирает устройство (CUDA, если есть, иначе CPU), переменная `device` выше нужна только для нашего собственного кода в задании 3. А `ExplainRunner` в Jupyter вызывается через `await runner.explain(...)`: синхронный `explain_sync` внутри уже работающего цикла событий Jupyter запрещен.

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
REVISION = '7ae557604adf67be50417f59c2c2f167def9a775'
SYSTEM = 'Answer with one word.'
PROMPT = 'What is the capital city of France?'

snapshot = ConversationSnapshot.from_turns((Turn(messages=(
    Message(role=Role.SYSTEM, text=SYSTEM),
    Message(role=Role.USER, text=PROMPT),
)),))


def make_config(estimator='exact', value_fn='logprob', evaluations=None, seed=0):
    # exact перебирает все 2^n коалиций; приближенным оценщикам задаем число прогонов evaluations
    budget = {'fraction': 1.0} if evaluations is None else {'evaluations': evaluations}
    return ExplainConfig.model_validate({
        'backend': {'kind': 'transformers', 'model_id': MODEL_ID,
                    'options': {'revision': REVISION, 'dtype': 'float32'}},
        'generation': {'max_new_tokens': 6, 'temperature': 0.0, 'top_k': 1},
        'explanation': {'estimator': estimator, 'value_fn': value_fn, 'budget': budget, 'seed': seed,
                        'archive': {'path': ''},                            # архив прогонов на диск не пишем
                        'plugin_options': {'tokens': {'roles': ['user']}}},  # игроки — только слова вопроса
    })

Дальше мы много раз объясним один и тот же промпт: разными оценщиками, с разными бюджетами и сидами, с разными функциями ценности. Каждый запуск заново генерировал бы ответы на те же урезанные промпты. Поэтому обернем бэкенд в кэш: ответ на каждый урезанный промпт генерируется один раз, а повторные запросы берутся из словаря `backend.records`. Генерация жадная, так что ответ из кэша совпадает с тем, что дала бы модель при повторном вызове. Обертка заодно всегда просит у бэкенда эмбеддинги ответа: они нужны функции ценности `embedding_cosine`, а остальным не мешают.

`ExplainRunner` принимает готовый бэкенд аргументом `backend=`; со стороны библиотеки это обычный объект с полем `model_id` и асинхронным методом `generate`.

In [ ]:
class CachedBackend:
    def __init__(self, inner):
        self.inner = inner
        self.records = {}      # (роль, текст) всех сообщений урезанного промпта -> ответ модели
        self.model_calls = 0   # сколько раз модель действительно генерировала

    @property
    def model_id(self):
        return self.inner.model_id

    @property
    def returns_logprobs(self):
        return True

    async def generate(self, snap, generation):
        key = tuple((m.role.value, m.text) for m in snap.messages())
        if key not in self.records:
            self.model_calls += 1
            self.records[key] = await self.inner.generate(
                snap, generation.model_copy(update={'compute_embeddings': True}))
        return self.records[key]


backend = CachedBackend(TransformersTextBackend(PluginContext(config=make_config())))


async def explain(**kwargs):
    return await ExplainRunner(make_config(**kwargs), backend=backend).explain(snapshot)

## Точный оценщик и эффективность

Первая функция ценности — `logprob`. Для коалиции $S$ модель отвечает на промпт, в котором остались только слова из $S$, и $v(S)$ — средняя лог-вероятность токенов этого ответа:

$$v(S) = \frac{1}{|g(x_S)|}\sum_{t} \log p\big(g_t(x_S)\mid x_S, g_{<t}(x_S)\big),$$

где $g(x_S)$ — ответ модели на урезанный промпт $x_S$. Обратите внимание: это уверенность модели в **своем собственном** ответе на урезанный промпт, а не вероятность ответа «Paris». Ответ на каждую коалицию свой, поэтому $v(S)$ говорит, насколько уверенно модель отвечает хоть что-то. Вероятность конкретного ответа зададим сами в задании 3.

Оценщик `exact` перебирает все $2^7=128$ коалиций — на CPU Colab это занимает около минуты-двух. Затем значения Шепли считаются по классической формуле.

In [ ]:
exact = await explain(estimator='exact', value_fn='logprob')
res = exact.result
words = [span.text for span in exact.player_set.spans]
n = len(words)
phi_exact = np.array(res.values)

print('игроки:', words)
print('вызовов модели:', backend.model_calls, '| ответ на полный промпт:', repr(backend.records[(('system', SYSTEM), ('user', PROMPT))].text))
print(f'v(пустой) = {res.base_value:.4f}, v(N) = {res.grand_value:.4f}')
pd.DataFrame({'phi (logprob)': phi_exact}, index=words).round(4)

Результат хранит не только вклады `values`, но и обе границы игры: `base_value` $=v(\varnothing)$ и `grand_value` $=v(N)$. Это позволяет проверить аксиому эффективности одной строкой.

In [ ]:
print(f'сумма phi        = {phi_exact.sum():.6f}')
print(f'v(N) - v(пустой) = {res.grand_value - res.base_value:.6f}')
print('эффективность выполняется:', np.isclose(phi_exact.sum(), res.grand_value - res.base_value))

In [ ]:
def plot_words(values, title, xlabel):
    colors = ['tab:red' if v > 0 else 'tab:blue' for v in values]
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.bar(words, values, color=colors)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel(xlabel)
    ax.set_title(title)
    plt.show()


plot_words(phi_exact, 'Точные значения Шепли слов промпта, value_fn = logprob',
           'phi: вклад в среднюю\nлог-вероятность ответа')

Красные столбики повышают уверенность модели в ответе, синие — понижают. Посмотрим и на сами ответы: что модель говорит, когда от вопроса остается одно слово.

In [ ]:
def masked_snapshot(present):
    # тот же урезанный промпт, который библиотека отдает модели для коалиции present
    return DeletePolicy().apply(snapshot, exact.player_set, CoalitionMask(tuple(present)))


def record_of(present):
    return backend.records[tuple((m.role.value, m.text) for m in masked_snapshot(present).messages())]


for i in range(n):
    present = [j == i for j in range(n)]
    print(f'{{{words[i]}}}'.ljust(12), '->', repr(record_of(present).text))
print('{}'.ljust(12), '->', repr(record_of([False] * n).text))

**Вопрос для самопроверки.** Какое слово получило наибольшее значение Шепли в игре `logprob`, и чему оно равно? Чему равны сумма значений Шепли и $v(N)-v(\varnothing)$?

<details><summary>Проверить себя</summary>

Наибольшее значение Шепли у слова `France?`: $\phi = 0.4479$, на втором месте `city` с $0.2985$. Сумма значений Шепли равна $1.0306$, и $v(N)-v(\varnothing) = -0.0530-(-1.0836) = 1.0306$: эффективность выполняется. Отрицательные вклады у `the` ($-0.0263$) и `of` ($-0.0144$).

</details>

## Задание 1. Значения Шепли по таблице $v(S)$

Все 128 ответов уже лежат в кэше `backend.records`, а функция `record_of(present)` достает ответ для любой коалиции. Ниже по ним собрана таблица `v_logprob`: ключ — множество номеров слов в коалиции (`frozenset`), значение — средняя лог-вероятность ответа, ровно как ее считает `value_fn='logprob'` (пустой ответ дает 0).

Напишите функцию `shapley_exact(v, n)`, которая по такой таблице считает значения Шепли всех $n$ игроков по формуле

$$\phi_i = \sum_{S\subseteq N\setminus i}\frac{|S|!\,(n-|S|-1)!}{n!}\,\big(v(S\cup i)-v(S)\big),$$

и сравните результат с `phi_exact` из библиотеки. Функция пригодится еще раз в задании 3.

In [ ]:
def mean_logprob(record):
    return sum(record.logprobs) / len(record.logprobs) if record.logprobs else 0.0


v_logprob = {frozenset(S): mean_logprob(record_of([i in S for i in range(n)]))
             for size in range(n + 1) for S in combinations(range(n), size)}
print('коалиций в таблице:', len(v_logprob))

In [ ]:
def shapley_exact(v, n):
    # Ваш код здесь: для каждого игрока i переберите коалиции S без i (combinations),
    # сложите вес |S|!(n-|S|-1)!/n!, умноженный на v[S | {i}] - v[S]
    ...


phi_manual = shapley_exact(v_logprob, n)
print(pd.DataFrame({'вручную': phi_manual, 'mllm-shap exact': phi_exact}, index=words).round(4))
print('максимум |разности|:', np.abs(phi_manual - phi_exact).max())

**Вопрос для самопроверки.** Какого порядка максимум модуля разности между вашими значениями и значениями `exact`? О чем это говорит?

<details><summary>Проверить себя</summary>

Максимум модуля разности около $10^{-16}$ (в нашем прогоне $1.1\cdot10^{-16}$), то есть на уровне округления чисел с плавающей точкой. Оценщик `exact` — это та же классическая формула Шепли, примененная к той же таблице из 128 значений $v(S)$; разница только в порядке сложения.

</details>

## Задание 2. Приближенные оценщики при разных бюджетах

В `mllm-shap` три приближенных оценщика. Бюджет `evaluations` — общее число прогонов модели, пустая и полная коалиции входят в него.

- **`mc`** — Монте-Карло: случайные коалиции, вклад слова — разность средних $v$ «со словом» и «без слова» с весами Шепли.
- **`complementary`** — комплементарные вклады (Zhang и др., 2023): коалиции берутся парами $S$ и $N\setminus S$, и одна пара $CC(S)=v(S)-v(N\setminus S)$ обновляет оценки всех $n$ игроков сразу. Минимальный бюджет $2n+2=16$: пара $(N,\varnothing)$ и пары $(\{i\}, N\setminus\{i\})$ для каждого слова.
- **`neyman_cc`** — тот же оценщик с двухфазной схемой из SGPA: в фазе 1 небольшая равная выборка по стратам (размерам коалиций) оценивает разброс $\sigma_{i,j}$, в фазе 2 остаток бюджета раздается стратам пропорционально $\sigma$.

Сравним их с точным ответом `phi_exact`. Для каждого оценщика, каждого бюджета из `budgets` и каждого сида из `seeds` запустите `await explain(estimator=..., value_fn='logprob', evaluations=..., seed=...)` и посчитайте две величины: среднюю абсолютную ошибку $\frac1n\sum_i|\hat\phi_i-\phi_i|$ и корреляцию Пирсона между $\hat\phi$ и `phi_exact`. Усредните обе по сидам и сложите в таблицу `errors` со столбцами `estimator`, `budget`, `MAE`, `pearson`.

Функция ценности здесь именно `logprob`: значение $v(S)$ зависит только от ответа на коалицию $S$. У `tfidf_cosine` это не так — веса IDF подгоняются по всем ответам одного запуска, и у приближенного оценщика, который видел только часть коалиций, получилась бы немного другая игра.

Модель новых ответов почти не генерирует: все 128 коалиций уже в кэше, поэтому ячейка идет секунды. Бюджет ограничен числом коалиций, $2^7=128$.

In [ ]:
estimators = ['mc', 'complementary', 'neyman_cc']
budgets = [20, 32, 48, 64, 96, 128]
seeds = range(10)

rows = []
# Ваш код здесь: тройной цикл по estimators, budgets и seeds; для каждой пары (оценщик, бюджет)
# добавьте в rows словарь {'estimator', 'budget', 'MAE', 'pearson'} со средними по сидам
errors = pd.DataFrame(rows)
print('вызовов модели всего:', backend.model_calls)
errors.pivot(index='budget', columns='estimator', values=['MAE', 'pearson']).round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for estimator in estimators:
    part = errors[errors.estimator == estimator]
    axes[0].plot(part.budget, part.MAE, marker='o', label=estimator)
    axes[1].plot(part.budget, part.pearson, marker='o', label=estimator)
axes[0].set_ylabel('средняя абсолютная ошибка phi ↓')
axes[1].set_ylabel('корреляция Пирсона с exact ↑')
for ax in axes:
    ax.set_xlabel('бюджет: число прогонов модели')
    ax.axvline(2 ** n, color='gray', linestyle='--', linewidth=0.8)
    ax.legend()
fig.suptitle(f'Приближенные оценщики против exact, value_fn = logprob, среднее по {len(seeds)} сидам; '
             f'пунктир — 2^n = {2 ** n} коалиций')
plt.tight_layout()
plt.show()
print(f'бюджет SGPA 3n^2 = {3 * n ** 2}, полный перебор 2^n = {2 ** n}')

**Вопрос для самопроверки.** Какой оценщик дает наименьшую среднюю абсолютную ошибку при бюджете 32 прогона и чему она равна? Что происходит с комплементарными оценщиками при бюджете 128? И сравните бюджет SGPA $3n^2$ с полным перебором $2^n$ для наших 7 слов: когда приближенный оценщик вообще имеет смысл?

<details><summary>Проверить себя</summary>

При бюджете 32 прогона наименьшая средняя абсолютная ошибка у `complementary`: $0.0738$ (у `neyman_cc` $0.0807$, у `mc` $0.1454$; среднее по 10 сидам). Неймановское распределение выигрывает у `complementary` при бюджетах 20 и 64: $0.0827$ против $0.1003$ и $0.0401$ против $0.0420$. При бюджете 128 у всех трех оценщиков ошибка 0 и корреляция 1: 128 = $2^7$, они видят все коалиции. Бюджет SGPA $3n^2 = 147$ для 7 слов больше полного перебора $2^7 = 128$, так что здесь точный оценщик выгоднее. Приближенный оценщик имеет смысл, когда $3n^2 < 2^n$, то есть начиная с 8 игроков ($192 < 256$); при 10 словах это 300 прогонов против 1024.

</details>

## Выбор функции ценности — выбор игры

Урок «SHAP на генеративных моделях» делит функции ценности на две стратегии: **похожесть** ответа на урезанный промпт на ответ на полный и **вероятность**. В `mllm-shap` есть обе:

- `tfidf_cosine` — косинус TF-IDF-векторов ответа на коалицию и ответа на полный промпт, как в TokenSHAP; по построению $v(N)=1$;
- `embedding_cosine` — косинус усредненных входных эмбеддингов токенов двух ответов, как в PixelSHAP;
- `logprob` — средняя лог-вероятность собственного ответа, ее мы уже посчитали.

Точным оценщиком посчитаем значения Шепли тех же 7 слов во всех трех играх. Все ответы уже в кэше, модель больше не вызывается: меняется только то, как ответы превращаются в число $v(S)$.

In [ ]:
games = {'logprob': res}
for value_fn in ['tfidf_cosine', 'embedding_cosine']:
    games[value_fn] = (await explain(estimator='exact', value_fn=value_fn)).result
print('вызовов модели всего:', backend.model_calls)

table = pd.DataFrame({name: r.values for name, r in games.items()}, index=words)
bounds = pd.DataFrame({name: [r.base_value, r.grand_value, sum(r.values)] for name, r in games.items()},
                      index=['v(пустой)', 'v(N)', 'сумма phi'])
print(bounds.round(4), '\n')
table.round(4)

Значения в трех играх измеряются в разных единицах, поэтому для графика поделим каждый столбец на его $v(N)-v(\varnothing)$: тогда столбики показывают долю всего выигрыша игры, которую получило слово, и в каждой игре в сумме дают 1.

In [ ]:
share = table / bounds.loc['v(пустой)'].rsub(bounds.loc['v(N)'])
pos = np.arange(n)
fig, ax = plt.subplots(figsize=(10, 4))
for k, name in enumerate(share.columns):
    ax.bar(pos + (k - 1) * 0.27, share[name], width=0.27, label=name)
ax.set_xticks(pos, words)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('phi / (v(N) - v(пустой))')
ax.set_title('Доля выигрыша игры у каждого слова в трех играх (exact)')
ax.legend(title='value_fn')
plt.show()
print(share.round(3), '\n')

rho = pd.DataFrame([[spearmanr(table[a], table[b])[0] for b in table] for a in table], index=table.columns, columns=table.columns)
print('Корреляция Спирмена рангов слов между играми:')
rho.round(3)

Эффективность выполняется во всех трех играх: сумма значений Шепли равна $v(N)-v(\varnothing)$. Но смысл этой суммы разный. В `tfidf_cosine` и `embedding_cosine` делится похожесть на ответ «Paris»: слово получает тем больше, чем сильнее оно держит ответ похожим на полный. В `logprob` делится уверенность модели в том, что она отвечает. Это три разные игры на одних и тех же 128 ответах.

**Вопрос для самопроверки.** Какое слово получает наибольшую долю выигрыша в игре `tfidf_cosine` и какое в игре `logprob`? Какая пара игр сильнее всего расходится по рангам слов (наименьшая корреляция Спирмена)?

<details><summary>Проверить себя</summary>

В игре `tfidf_cosine` наибольшая доля у `France?` ($0.644$), в игре `logprob` тоже у `France?` ($0.435$), а в игре `embedding_cosine` — у `capital` ($0.273$; у `France?` там $0.223$). Сильнее всего расходятся `logprob` и `embedding_cosine`: корреляция Спирмена рангов слов $0.179$. Ближе всего две игры похожести, `tfidf_cosine` и `embedding_cosine`: $0.857$.

</details>

## Задание 3. Своя функция ценности: вероятность ответа «Paris»

Ни одна из встроенных функций ценности не спрашивает, насколько вероятен **конкретный** ответ. Зададим вероятностную игру из урока «SHAP на генеративных моделях», шаг «Постановки через вероятность»:

$$v(S) = \log p\big(\text{«Paris»} \mid x_S\big) = \sum_t \log p\big(y_t \mid x_S, y_{<t}\big),$$

где $y$ — токены ответа на полный промпт. Здесь модель ничего не генерирует: ответ подставляется целиком, и за один прямой проход мы читаем вероятность каждого его токена (teacher forcing).

Загрузите ту же модель через `transformers` и напишите функцию `target_logprob(user_text)`: соберите промпт шаблоном чата (`tokenizer.apply_chat_template(..., add_generation_prompt=True)`) из системного сообщения и урезанного вопроса, допишите токены ответа `TARGET`, сделайте прямой проход и сложите лог-вероятности токенов ответа. Урезанный вопрос для коалиции дает `masked_snapshot(present).messages()[1].text`. Затем соберите таблицу `v_target` по всем 128 коалициям, посчитайте значения Шепли своей функцией `shapley_exact` из задания 1 и проверьте эффективность.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
lm = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION, dtype=torch.float32).to(device).eval()
TARGET = backend.records[(('system', SYSTEM), ('user', PROMPT))].text
print('целевой ответ:', repr(TARGET))

In [ ]:
target_ids = tokenizer(TARGET, add_special_tokens=False, return_tensors='pt').input_ids.to(device)


def target_logprob(user_text):
    # Ваш код здесь: шаблон чата -> токены промпта, к ним target_ids, прямой проход lm под torch.no_grad(),
    # сумма log_softmax по токенам ответа (логит позиции t предсказывает токен t + 1)
    ...


v_target = # Ваш код здесь: словарь frozenset(S) -> target_logprob(урезанный вопрос) по всем коалициям
phi_target = shapley_exact(v_target, n)

v_empty, v_full = v_target[frozenset()], v_target[frozenset(range(n))]
print(f'log p(ответ | пустой вопрос) = {v_empty:.4f}, p = {np.exp(v_empty):.4f}')
print(f'log p(ответ | полный вопрос) = {v_full:.4f}, p = {np.exp(v_full):.4f}')
print(f'p(ответ | полный) / p(ответ | пустой) = {np.exp(v_full - v_empty):.0f}')
print(f'сумма phi = {phi_target.sum():.4f}, v(N) - v(пустой) = {v_full - v_empty:.4f}')
pd.DataFrame({'phi: log p(Paris)': phi_target}, index=words).round(4)

In [ ]:
plot_words(phi_target, 'Точные значения Шепли слов промпта, v(S) = log p(Paris | x_S)',
           'phi: вклад в\nlog p(Paris)')
print('Спирмен с играми библиотеки:', {name: round(float(spearmanr(phi_target, table[name])[0]), 3) for name in table})

**Вопрос для самопроверки.** Во сколько раз полный вопрос повышает вероятность ответа «Paris» по сравнению с пустым? Какое слово дает наибольший вклад в $\log p(\text{Paris})$, и с какой из трех игр библиотеки эта игра согласуется лучше всего по рангам слов?

<details><summary>Проверить себя</summary>

Вероятность ответа «Paris» растет с $0.0001$ ($\log p = -8.9051$) при пустом вопросе до $0.9891$ ($\log p = -0.0110$) при полном, примерно в 7289 раз. Наибольший вклад у `France?`: $\phi = 5.7210$; у слова `is` вклад отрицательный, $-2.3204$. Сумма значений Шепли $8.8942 = v(N)-v(\varnothing)$. По рангам слов эта игра лучше всего согласуется с `tfidf_cosine`: корреляция Спирмена $0.964$ (с `embedding_cosine` $0.821$, с `logprob` $0.571$).

</details>

## Аудио: SGPA режет речь на слова

В версии 0.3.0 из аудиочасти SGPA в библиотеке есть разбиение записи на слова-игроки и маска-тишина: разбиение `sgpa_segments` (CTC-выравнивание моделью `facebook/wav2vec2-base-960h` плюс уточнение границ по энергии и spectral flux с весами 0.8 и 0.2) и политики отсутствия, которые вырезают или заглушают отсчеты выключенных слов. Модели, которая отвечала бы на аудио, нет: бэкенды `transformers`, `lmstudio` и `api` отказываются принимать аудиосообщения, а `mock` ответа на звук не строит (это открытая задача D02 в репозитории библиотеки). Поэтому значения Шепли для аудио-LLM в этой версии не посчитать, а выравнивание и маскирование посмотрим.

Запись — фраза «I had that curiosity beside me at this moment» из корпуса VOiCES (лицензия CC BY 4.0); копия из репозитория библиотеки (тег v0.3.0) лежит в папке `data/` репозитория курса. Модель выравнивания весит около 360 МБ.

In [ ]:
WAV_URL = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/voices_sp0307.wav'
CTC_MODEL, CTC_REVISION = 'facebook/wav2vec2-base-960h', '22aad52d435eb6dbaf354bdad9b0da84ce7d6156'
TRANSCRIPT = 'I had that curiosity beside me at this moment'

urllib.request.urlretrieve(WAV_URL, 'voices_sp0307.wav')
audio = AudioPart(open('voices_sp0307.wav', 'rb').read(), sample_rate=16000)
aligner = SpectrogramGuidedAligner(Wav2Vec2Ctc(CTC_MODEL, revision=CTC_REVISION), energy_weight=0.8, flux_weight=0.2)
segments = aligner.align(TRANSCRIPT, audio)

pd.DataFrame([{'слово': s.token, 'начало CTC, с': s.ctc_start_time, 'начало, с': s.start_time,
               'конец, с': s.end_time, 'уверенность CTC': s.confidence, 'граница уточнена': s.boundary_refined}
              for s in segments]).round(3)

На рисунке библиотеки пунктир — грубые границы CTC, стрелки — куда их сдвинуло уточнение, сплошные линии — итоговые границы слов, нижняя панель — энергия RMS: разрезы попадают в ее провалы. Подписи на рисунке: `amplitude` — амплитуда, `RMS` — энергия кадра, `time (s)` — время в секундах, в легенде `segment` — отрезок слова, `CTC edge` — грубая граница CTC, `boundary` — итоговая граница.

In [ ]:
plot_audio_segments(audio, segments)
plt.show()

Теперь маска. Сообщение с аудио и транскриптом становится игроками-словами через `SgpaSegmentPartitioner`; у каждого игрока есть и отрезок текста, и отрезок отсчетов. Выключим каждое второе слово двумя политиками: `delete` вырезает отсчеты (запись становится короче), `neutral` заменяет их нулевой амплитудой и сохраняет тайминг остальных слов — это и есть маска-тишина из урока. В тексте транскрипта `delete` убирает слово, а `neutral` заменяет его подчеркиваниями той же длины.

In [ ]:
spoken = ConversationSnapshot.from_turns((Turn(messages=(Message(role=Role.USER, content=(TextPart(TRANSCRIPT), audio)),)),))
audio_players = SgpaSegmentPartitioner(aligner=aligner).partition(spoken)
keep = CoalitionMask(tuple(i % 2 == 0 for i in range(audio_players.num_players)))

waves = {}
for policy in (DeletePolicy(), NeutralPolicy()):
    rendered = policy.apply(spoken, audio_players, keep).messages()[0]
    waves[policy.name], rate = decode_audio(rendered.content[1].data())
    print(f'{policy.name:8} {len(waves[policy.name]) / rate:.2f} с | текст: {rendered.text!r}')

original, rate = decode_audio(audio.data())
print(f'исходная запись: {len(original) / rate:.2f} с')
fig, axes = plt.subplots(3, 1, figsize=(10, 5), sharex=True)
for ax, (title, wave) in zip(axes, [('исходная запись', original), ('neutral: тишина вместо слов', waves['neutral']),
                                    ('delete: слова вырезаны', waves['delete'])]):
    ax.plot(np.arange(len(wave)) / rate, wave, linewidth=0.4)
    ax.set_title(title, fontsize=10)
axes[-1].set_xlabel('время, с')
plt.tight_layout()
plt.show()

**Вопрос для самопроверки.** Сколько слов транскрипта получили отрезки и у скольких из них обе границы уточнены по энергии и spectral flux? Сколько секунд длится исходная запись и сколько — после политики `delete`?

<details><summary>Проверить себя</summary>

Отрезки получили все 9 слов транскрипта, обе границы уточнены у 7 из них; у `curiosity` и `beside` общая граница не уточнена: в окне поиска не нашлось достаточно тихой точки, и разрез остался в середине промежутка между словами по CTC. Исходная запись длится 3.40 с; после `delete` с выключенными словами `had`, `curiosity`, `me`, `this` остается 1.92 с, а после `neutral` длина та же, 3.40 с.

</details>

## Картинка: PixelSHAP своими руками

В уроке «SHAP на генеративных моделях» PixelSHAP (Goldshmidt, 2025) переносит TokenSHAP на картинки: игроки — объекты картинки, «убрать» объект значит закрасить его область, а функция ценности — похожесть ответов:

$$v(S) = \cos\big(E(y),\ E(y_S)\big),$$

где $y$ — ответ VLM на исходную картинку, $y_S$ — ответ на картинку, где остались только объекты из $S$, а $E$ — эмбеддинг текста. В `mllm-shap` 0.3.0 картинок нет, а пакет авторов `TokenSHAP` версии 0.2.2 на PyPI PixelSHAP не содержит: в репозитории авторов он ждет GroundingDINO, SAM и модель через Ollama или платный API. Поэтому соберем ту же игру сами из трех маленьких открытых моделей, которым не нужны ни ключ API, ни токен Hugging Face:

- сегментация `facebook/mask2former-swin-tiny-coco-panoptic` вместо пары GroundingDINO и SAM: она сразу дает маски объектов с метками COCO;
- VLM `HuggingFaceTB/SmolVLM-256M-Instruct` отвечает на вопрос о картинке;
- `sentence-transformers/all-MiniLM-L6-v2` дает эмбеддинги ответов для косинуса.

Картинка — `data/cat_and_dog.jpg` из репозитория курса: кошка и собака на скамейке у стены. Рядом с игрой похожести посчитаем и вероятностную игру, как в задании 3: $v(S) = \log p(y \mid \text{картинка}_S)$ — лог-вероятность, которую VLM дает своему полному ответу $y$ на урезанной картинке.

Автор фото кошки и мопса: [Priscilla Du Preez, Unsplash](https://unsplash.com/photos/qtkpOxCpl0Y), лицензия Unsplash.

In [ ]:
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor, pipeline
from sentence_transformers import SentenceTransformer

urllib.request.urlretrieve('https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/cat_and_dog.jpg',
                           'cat_and_dog.jpg')
image = Image.open('cat_and_dog.jpg').convert('RGB')
image.thumbnail((768, 768))                    # 768 x 512 вместо 5472 x 3648: маски и VLM считаются быстрее

segmenter = pipeline('image-segmentation', model='facebook/mask2former-swin-tiny-coco-panoptic',
                     revision='df6b1142ff50c3276559d9d78f35f6a579c75a77', device=device)
segments = segmenter(image)
objects = [segment['label'] for segment in segments]
masks = [np.array(segment['mask']) > 0 for segment in segments]
n_objects = len(objects)
for segment, mask in zip(segments, masks):
    print(f'{segment["label"]:<18} уверенность {segment["score"]:.3f}, доля пикселей {mask.mean():.3f}')
print(f'пикселей вне объектов: {1 - np.logical_or.reduce(masks).mean():.3f}')

In [ ]:
fig, axes = plt.subplots(1, n_objects + 1, figsize=(3.2 * (n_objects + 1), 2.6))
axes[0].imshow(image)
axes[0].set_title('картинка')
for ax, name, mask in zip(axes[1:], objects, masks):
    ax.imshow(mask, cmap='gray')
    ax.set_title(f'игрок: {name}')
for ax in axes:
    ax.axis('off')
plt.show()

Сегментатор нашел четыре объекта, и они покрывают всю картинку: собаку (`dog`), кошку (`cat`), скамейку вместе с пледом (метка COCO `chair`) и стену (`wall-other-merged`). Это и есть игроки, всего $2^4 = 16$ коалиций, поэтому значения Шепли посчитаем точно. Выключенный объект закрашиваем серым, так что при пустой коалиции VLM видит серый прямоугольник.

Вопрос к VLM один на все коалиции. Генерация жадная, поэтому ответ на каждую урезанную картинку однозначен.

In [ ]:
VLM_ID = 'HuggingFaceTB/SmolVLM-256M-Instruct'
VLM_REVISION = '7e3e67edbbed1bf9888184d9df282b700a323964'
QUESTION = 'What animals are in this picture? Answer in one short sentence.'

processor = AutoProcessor.from_pretrained(VLM_ID, revision=VLM_REVISION)
vlm = AutoModelForImageTextToText.from_pretrained(VLM_ID, revision=VLM_REVISION, dtype=torch.float32).to(device).eval()
vlm_prompt = processor.apply_chat_template(
    [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': QUESTION}]}], add_generation_prompt=True)
pixels = np.array(image)


def masked_image(present):
    # объекты вне коалиции закрашиваются серым, остальная картинка не меняется
    result = pixels.copy()
    for keep, mask in zip(present, masks):
        if not keep:
            result[mask] = 128
    return Image.fromarray(result)


def vlm_inputs(picture):
    return processor(text=vlm_prompt, images=[picture], return_tensors='pt').to(device)


def vlm_answer_ids(picture):
    inputs = vlm_inputs(picture)
    with torch.no_grad():
        output = vlm.generate(**inputs, max_new_tokens=30, do_sample=False)
    return output[0, inputs['input_ids'].shape[1]:]


full_answer_ids = vlm_answer_ids(image)
FULL_ANSWER = processor.decode(full_answer_ids, skip_special_tokens=True).strip()
print('ответ на полную картинку:', repr(FULL_ANSWER))

Теперь пройдем все 16 коалиций. Для каждой VLM отвечает на урезанную картинку, и мы считаем обе функции ценности: косинус эмбеддингов этого ответа и полного ответа и лог-вероятность полного ответа на урезанной картинке (ответ подставляется целиком, один прямой проход, как в задании 3). На CPU Colab проход занимает около полутора минут.

In [ ]:
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2',
                               revision='1110a243fdf4706b3f48f1d95db1a4f5529b4d41', device=device)
full_embedding = embedder.encode([FULL_ANSWER], normalize_embeddings=True)[0]
special_ids = set(processor.tokenizer.all_special_ids)
answer_target = torch.tensor([[t for t in full_answer_ids.tolist() if t not in special_ids]], device=device)


def full_answer_logprob(picture):
    inputs = vlm_inputs(picture)
    ids = torch.cat([inputs['input_ids'], answer_target], dim=1)
    with torch.no_grad():
        logits = vlm(input_ids=ids, attention_mask=torch.ones_like(ids), pixel_values=inputs['pixel_values'],
                     pixel_attention_mask=inputs['pixel_attention_mask']).logits
    logits = logits[0, inputs['input_ids'].shape[1] - 1:-1].float()  # прогноз каждого токена ответа
    return logits.log_softmax(-1).gather(1, answer_target[0][:, None]).sum().item()


pixel_records = {}
for size in range(n_objects + 1):
    for S in combinations(range(n_objects), size):
        picture = masked_image([i in S for i in range(n_objects)])
        text = processor.decode(vlm_answer_ids(picture), skip_special_tokens=True).strip()
        similarity = float(embedder.encode([text], normalize_embeddings=True)[0] @ full_embedding)
        pixel_records[frozenset(S)] = (text, similarity, full_answer_logprob(picture))

pd.DataFrame([{'объекты на картинке': ', '.join(objects[i] for i in sorted(S)) or '(ничего)', 'ответ VLM': text,
               'v: похожесть': similarity, 'v: log p(полный ответ)': logprob}
              for S, (text, similarity, logprob) in pixel_records.items()]).round(4)

Значения Шепли обеих игр посчитаем функцией `shapley_exact` из задания 1. Рядом поставим приближение первого порядка, на которое опирается PixelSHAP: вклад объекта — насколько меняется $v$, если убрать из полной картинки только его, $v(N) - v(N\setminus i)$. Оно стоит $n+1$ вызов VLM вместо $2^n$. В уроке приведена оценка авторов: первый порядок дает «92% корреляции» с точными значениями Шепли, без указания картинок и меры. Проверим ее на нашей картинке.

In [ ]:
v_similarity = {S: record[1] for S, record in pixel_records.items()}
v_logprob_image = {S: record[2] for S, record in pixel_records.items()}
everything = frozenset(range(n_objects))


def first_order(v):
    return np.array([v[everything] - v[everything - {i}] for i in range(n_objects)])


pixel_table = pd.DataFrame({
    'phi: похожесть': shapley_exact(v_similarity, n_objects),
    'первый порядок: похожесть': first_order(v_similarity),
    'phi: log p': shapley_exact(v_logprob_image, n_objects),
    'первый порядок: log p': first_order(v_logprob_image),
}, index=objects)
print(pixel_table.round(4).to_string(), '\n')
for name, v, column in [('похожесть', v_similarity, 'phi: похожесть'), ('log p', v_logprob_image, 'phi: log p')]:
    print(f'{name}: сумма phi = {pixel_table[column].sum():.4f}, v(N) - v(пустая) = {v[everything] - v[frozenset()]:.4f}, '
          f'Пирсон phi и первого порядка = {pearsonr(pixel_table[column], first_order(v))[0]:.3f}')

Раскрасим объекты картинки их значениями Шепли: красный повышает $v$, синий понижает, шкала у каждой игры своя.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, column, title in [(axes[0], 'phi: похожесть', 'PixelSHAP: v(S) = cos(E(y), E(y_S))'),
                          (axes[1], 'phi: log p', 'v(S) = log p(y | картинка_S)')]:
    heat = np.zeros(pixels.shape[:2])
    for value, mask in zip(pixel_table[column], masks):
        heat[mask] = value
    limit = np.abs(pixel_table[column]).max()
    ax.imshow(image)
    shown = ax.imshow(heat, cmap='RdBu_r', vmin=-limit, vmax=limit, alpha=0.6)
    ax.set_title(title)
    ax.axis('off')
    fig.colorbar(shown, ax=ax, fraction=0.03, label='значение Шепли объекта')
plt.show()

**Вопрос для самопроверки.** Какой объект получил наибольшее значение Шепли в игре похожести и чему оно равно? Каков вклад скамейки (`chair`) в этой игре и почему первый порядок его не видит? Чему равна корреляция Пирсона точных значений и первого порядка в каждой игре и что из этого можно сказать об оценке «92% корреляции»?

<details><summary>Проверить себя</summary>

В игре похожести наибольшее значение Шепли у кошки: $\phi = 0.2637$, у собаки 0.1806, у стены 0.1041. Скамейка (`chair`) получила $-0.0937$: картинку, где из объектов осталась только скамейка, VLM описывает как «Antelope.», а скамейку с собакой — как «Pug.», и похожесть на ответ «Dogs and cats.» падает до 0.2607 и 0.2525. Первый порядок дает скамейке 0: если убрать из полной картинки только ее, ответ остается «Dogs and cats.», и $v(N)-v(N\setminus\text{chair}) = 0$. Вклад скамейки виден только в маленьких коалициях, а их первый порядок не смотрит. Эффективность выполняется в обеих играх: сумма значений равна $v(N)-v(\varnothing)$ (0.4547 и 9.4595), но в игре похожести делится похожесть на полный ответ, а не сам ответ.

Корреляция Пирсона точных значений и первого порядка равна 0.929 в игре похожести и 0.739 в игре $\log p$. Близость первого числа к «92%» авторов оценку не подтверждает: это корреляция по четырем объектам одной картинки, на тех же 16 урезанных картинках в другой игре она 0.739, а в игре $\log p$ первый порядок завышает вклад собаки втрое (12.3492 против 3.9132). Числа получены на CPU; на GPU Colab последние знаки могут отличаться.

</details>

## Заключение

Мы проверили на `mllm-shap` 0.3.0 обе мысли уроков. Точный оценщик на 7 словах требует 128 прогонов модели, и сумма значений Шепли в каждой игре равна $v(N)-v(\varnothing)$. Комплементарные оценщики приближают точный ответ по части коалиций, но выигрывают только тогда, когда бюджет меньше $2^n$: бюджет SGPA $3n^2$ меньше полного перебора лишь начиная с 8 игроков ($3\cdot8^2=192 < 2^8=256$), а при 10 игроках это уже 300 прогонов против 1024. Смена функции ценности на тех же ответах модели меняет ранги слов, потому что меняет саму игру: похожесть на ответ, уверенность модели и вероятность конкретного ответа — три разных вопроса к модели. Для аудио библиотека режет речь на слова и умеет заменять их тишиной, а модели, которая отвечала бы на звук, в этой версии нет. На картинке PixelSHAP той же логикой делит между объектами похожесть ответа VLM: точные значения четырех объектов стоят 16 вызовов VLM, а приближение первого порядка за 5 вызовов не видит отрицательного вклада скамейки, который проявляется только в маленьких коалициях.

**Спасибо за работу!**